In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
PROJECT_DIR = Path('/content/drive/MyDrive/SafeVision_AI')  
assert PROJECT_DIR.exists(), f'Project folder not found: {PROJECT_DIR}'
%cd $PROJECT_DIR
print('Project:', PROJECT_DIR)

In [ ]:
!pip -q install -r requirements.txt

In [ ]:
import torch, ultralytics, platform
print('Python:', platform.python_version())
print('Ultralytics:', ultralytics.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('No GPU runtime detected. In Colab choose Runtime > Change runtime type > GPU.')

## 1. Train the baseline model (YOLO26n)


In [ ]:
from ultralytics import YOLO
from pathlib import Path

model_n = YOLO('yolo26n.pt')
train_n = model_n.train(
    data=str(PROJECT_DIR / 'data.yaml'),
    epochs=50,
    imgsz=640,
    batch=8,
    device=0,
    workers=2,
    patience=12,
    project=str(PROJECT_DIR / 'runs' / 'colab'),
    name='safevision_yolo26n',
    exist_ok=True,
    seed=42,
    pretrained=True,
    cos_lr=True,
    close_mosaic=10,
    plots=True,
    save=True,
)
BEST_N = Path(model_n.trainer.save_dir) / 'weights' / 'best.pt'
print('Best checkpoint:', BEST_N, 'exists:', BEST_N.exists())

## 2. Evaluate the baseline on validation


In [ ]:
val_n_model = YOLO(str(BEST_N))
val_n = val_n_model.val(data=str(PROJECT_DIR / 'data.yaml'), imgsz=640, device=0, plots=True)
metrics_n = {
    'model': 'YOLO26n',
    'precision_mean': float(val_n.box.mp),
    'recall_mean': float(val_n.box.mr),
    'mAP50': float(val_n.box.map50),
    'mAP50_95': float(val_n.box.map),
}
metrics_n

## 3. Optional model comparison (YOLO26s)



In [ ]:
RUN_SMALL = True  # set False to skip if time or GPU memory is limited
metrics = [metrics_n]

if RUN_SMALL:
    model_s = YOLO('yolo26s.pt')
    train_s = model_s.train(
        data=str(PROJECT_DIR / 'data.yaml'),
        epochs=50,
        imgsz=640,
        batch=4,
        device=0,
        workers=2,
        patience=12,
        project=str(PROJECT_DIR / 'runs' / 'colab'),
        name='safevision_yolo26s',
        exist_ok=True,
        seed=42,
        pretrained=True,
        cos_lr=True,
        close_mosaic=10,
        plots=True,
        save=True,
    )
    BEST_S = Path(model_s.trainer.save_dir) / 'weights' / 'best.pt'
    val_s_model = YOLO(str(BEST_S))
    val_s = val_s_model.val(data=str(PROJECT_DIR / 'data.yaml'), imgsz=640, device=0, plots=True)
    metrics_s = {
        'model': 'YOLO26s',
        'precision_mean': float(val_s.box.mp),
        'recall_mean': float(val_s.box.mr),
        'mAP50': float(val_s.box.map50),
        'mAP50_95': float(val_s.box.map),
    }
    metrics.append(metrics_s)

import pandas as pd
comparison = pd.DataFrame(metrics)
comparison

## 4. Select, export and download the checkpoint



In [ ]:
USE_SMALL = False
selected_weights = BEST_S if (RUN_SMALL and USE_SMALL) else BEST_N

import shutil, json
models_dir = PROJECT_DIR / 'models'
models_dir.mkdir(exist_ok=True)
final_weights = models_dir / 'best.pt'
shutil.copy2(selected_weights, final_weights)

results_dir = PROJECT_DIR / 'runs' / 'evaluation'
results_dir.mkdir(parents=True, exist_ok=True)
comparison.to_csv(results_dir / 'model_comparison_validation.csv', index=False)
(results_dir / 'colab_validation_metrics.json').write_text(json.dumps(metrics, indent=2))
print('Exported checkpoint:', final_weights, 'size MB:', round(final_weights.stat().st_size / 1024**2, 2))

In [ ]:
from google.colab import files
files.download(str(final_weights))